# SAMGA and visual LoRA on THINGS-EEG2

Train nine participants and evaluate the held-out participant, repeating the split across the selected subjects. Configure the experiment below, inspect the cached features, then run the final cell. Install the project dependencies before opening the notebook; cells never install packages.


In [ ]:
import torch

from things_eeg2.config import DataConfig, OptimizationConfig, RunConfig, resolve_paths
from things_eeg2.data import visual_feature_status
from things_eeg2.experiments import run_loso
from things_eeg2.model import EEGEncoderConfig, ModelConfig, SAMGAModel, VisualRouterConfig, count_parameters
from things_eeg2.paths import WORKSPACE_ROOT, workspace_path
from things_eeg2.score_calibration import ScoreCalibrationConfig
from things_eeg2.visual_lora import VisualLoRAConfig

print(f"Workspace: {WORKSPACE_ROOT}")
print(f"PyTorch {torch.__version__}; CUDA available: {torch.cuda.is_available()}")


## Configuration

`LOSS_MODE="samga"` uses diagonal-pair contrastive loss over the full batch. `LOSS_MODE="score"` makes all same-image pairs positives and optionally adds supervised support-to-query recovery. These are separate training configurations.

Each image is shared by all nine training participants in a batch. The visual encoder processes each unique image once. `VISUAL_LAYERS` selects intermediate ViT blocks; `VISUAL_POOLING` selects class-token or mean-patch features. LoRA can update either training stage independently. If both stage switches are disabled, cached visual features stay frozen. `LORA_ALPHA / LORA_RANK` scales the update; `LORA_MAX_UPDATE_RATIO` optionally caps its norm.

Calibration uses a common pool of training-concept images with known participant-specific EEG matches. The default recovery schedule reselects support pairs and refits maps from current embeddings every epoch. A pseudo-target participant is excluded from ordinary alignment in each recovery batch; its queries exclude the support pool. Target-subject test EEG never enters map fitting.

Relative paths resolve against `EEG2IMAGE_ROOT`, which defaults to the workspace containing this project.


In [ ]:
SCORE_POOL_CLASSES = 200
SCORE_LANDMARKS = 150
SCORE_SEED = 42
SCORE_CSLS_NEIGHBORS = 10
SCORE_RHO = 0.13

LOSS_MODE = "score"  
SCORE_ALIGNMENT_SCALE = 1.0  
SCORE_RECOVERY_WEIGHT = 0.2  
SCORE_RECOVERY_TEMPERATURE = 0.07
SCORE_RECOVERY_RHO = 0.13  
SCORE_RECOVERY_NEIGHBORS = 10  
SCORE_RECOVERY_LANDMARK_SCHEDULE = "per_epoch_v1"  
RUN_NAME_PREFIX = "score_loso" if LOSS_MODE == "samga" else "score_loss_loso"

VISUAL_LAYERS = (3, 5, 7, 9)  
VISUAL_POOLING = "cls"  
HELD_OUT_SUBJECTS = (1,2,3,4,5,6,7,8,9,10)
TOTAL_EPOCHS = 30
COARSE_EPOCHS = 15 
TRAIN_VISUAL_ENCODER_DURING_COARSE = True  
TRAIN_VISUAL_ENCODER_DURING_FINE = True  
PRIOR_CENTER = 9  
LORA_BLOCKS = (3, 4, 5, 6, 7, 8, 9)  
LORA_RANK = 192
LORA_ALPHA = 16.0  
LORA_MAX_UPDATE_RATIO = None  
LORA_TUNE_LAYER_NORM = True  
LORA_KEY = True  
LORA_OUTPUT = True  
LORA_MLP = True  
LORA_COARSE_LEARNING_RATE = 1e-4  
LORA_FINE_LEARNING_RATE = 1e-4    
RESUME_CHECKPOINT = None

config = RunConfig(
    data=DataConfig(
        dataset_root=workspace_path(r"training_data/nm000232"),
        visual_feature_dir=workspace_path(
            f"training_data/nm000232/stimuli/vit_b16_laion2b_layers_2_10_{VISUAL_POOLING}"
        ),
        held_out_subject=HELD_OUT_SUBJECTS[0],
        num_workers=0,
        pin_memory=True,
        evaluate_train_subjects_on_test=True,
        cross_subject_batches=True,
        images_per_batch=100,  
    ),
    model=ModelConfig(
        eeg=EEGEncoderConfig(
            channels=63,
            samples=250,
            temporal_filters=40,
            temporal_kernel=25,
            pooling_kernel=51,
            pooling_stride=5,
            dropout=0.5,
            feature_dim=1024,
        ),
        visual=VisualRouterConfig(
            layer_ids=VISUAL_LAYERS,
            raw_feature_dim=768,
            intermediate_dim=1024,
            alignment_dim=512,
            prior_center=PRIOR_CENTER,
            prior_strength=1,
            temperature=1.0,
            subject_dropout=0.3,
            layer_dropout=0.0,
            max_subject_id=10,
        ),
    ),
    optimization=OptimizationConfig(
        epochs=TOTAL_EPOCHS,
        coarse_epochs=COARSE_EPOCHS,
        batch_size=1024,  
        learning_rate=1e-4,
        fine_learning_rate=1e-4,
        weight_decay=1e-4,
        temperature=0.07,
        loss_mode=LOSS_MODE,
        score_alignment_scale=SCORE_ALIGNMENT_SCALE,
        score_recovery_weight=SCORE_RECOVERY_WEIGHT,
        score_recovery_temperature=SCORE_RECOVERY_TEMPERATURE,
        score_recovery_identity_ratio=SCORE_RECOVERY_RHO,
        score_recovery_csls_neighbors=SCORE_RECOVERY_NEIGHBORS,
        score_recovery_landmark_schedule=SCORE_RECOVERY_LANDMARK_SCHEDULE,
        mmd_weight_start=0.9,
        mmd_weight_end=0.5,
        mmd_kernel_scales=(0.1, 0.2, 0.5, 1.0, 2.0),
        smooth_kernel_size=5,
        smooth_channel_probability=0.3,
        frozen_smoothing=True,
        use_amp=True,
        seed=42,
    ),
    visual_lora=VisualLoRAConfig(
        enabled=TRAIN_VISUAL_ENCODER_DURING_COARSE or TRAIN_VISUAL_ENCODER_DURING_FINE,
        stage=("both" if TRAIN_VISUAL_ENCODER_DURING_COARSE and TRAIN_VISUAL_ENCODER_DURING_FINE
               else "coarse" if TRAIN_VISUAL_ENCODER_DURING_COARSE else "fine"),
        checkpoint=workspace_path(r"models/ViT-B-16-laion2b/open_clip_model.safetensors"),
        blocks=LORA_BLOCKS,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
        max_update_ratio=LORA_MAX_UPDATE_RATIO,
        tune_layer_norm=LORA_TUNE_LAYER_NORM,
        lora_key=LORA_KEY,
        lora_output=LORA_OUTPUT,
        lora_mlp=LORA_MLP,
        images_per_batch=100,  
        gradient_cache_microbatch_size=100,  
        evaluation_image_batch_size=100,
        learning_rate=LORA_FINE_LEARNING_RATE,
        coarse_learning_rate=LORA_COARSE_LEARNING_RATE,
        gradient_checkpointing=True,
        pooling=VISUAL_POOLING,
    ),
    score=ScoreCalibrationConfig(
        enabled=True,
        pool_class_count=SCORE_POOL_CLASSES,
        landmark_count=SCORE_LANDMARKS,
        seed=SCORE_SEED,
        regularization_ratio=SCORE_RHO,
        csls_neighbors=SCORE_CSLS_NEIGHBORS,
    ),
    checkpoint_root=workspace_path(r"checkpoints/things_eeg2_score_loso"),
    run_name=f"{RUN_NAME_PREFIX}_subject_01",
    resume_checkpoint=RESUME_CHECKPOINT,
)

config = resolve_paths(config)


## Cached visual features

Inspect the selected feature files and their extraction manifests. This cell does not encode images or start training. Run `python -m things_eeg2.extract_vit_b16_features` if caches are missing.


In [ ]:
status = visual_feature_status(config.data, config.model)
print(f"Visual features ready: {status['ready']}")
print(f"Present: {len(status['present'])} | missing: {len(status['missing'])}")
for path in status['missing']:
    print(f"  MISSING: {path}")
for issue in status['incomplete']:
    print(f"  INCOMPLETE: {issue}")
print("Expected train shape per layer:", status['expected_train_shape'])
print("Expected test shape per layer:", status['expected_test_shape'])


## Architecture and parameter count


In [ ]:
model = SAMGAModel(config.model)
total, trainable = count_parameters(model)
print(f"Parameters: total {total:,} | trainable {trainable:,}")
print("TSConv output time points:", model.eeg_encoder.output_time_points)
print("TSConv flattened dimension:", model.eeg_encoder.flattened_dim)
print("Initial global router weights:", model.visual_target.router.global_weights().detach())
print(model)


## Train the selected held-out folds

Compatible completed runs are skipped and interrupted runs resume from `last.pt`. Set `RESUME_CHECKPOINT` to extend a particular run by the configured coarse/fine schedule. Results include final-epoch metrics and independently selected epoch peaks; distinguish them when reporting accuracy.


In [ ]:
summary = run_loso(
    config, HELD_OUT_SUBJECTS, RUN_NAME_PREFIX, TOTAL_EPOCHS, COARSE_EPOCHS
)
